# Construir el modelo canónico

Notebook operativo transversal. Para cada ejecución, ajusta únicamente la celda **PARÁMETROS** y ejecuta el notebook completo. Las entidades, fuentes, mappings, validaciones y destinos se definen en los YAML del Lakehouse.

- `ENTITY_NAME = None`: todas las entidades habilitadas; escribe un nombre para limitar la ejecución.
- `EXECUTION_MODE = None`: toma el modo de cada entidad desde su YAML; también se puede elegir `FULL_LOAD`, `UPSERT` o `SCD2` como override temporal.
- `PRECHECK_ONLY = True`: no escribe. Cambia a `False` únicamente cuando hayas revisado el precheck.

Requiere el wheel instalado en el Environment y el Lakehouse del proyecto adjunto. `PRECHECK_ONLY = True` valida la configuración y las fuentes sin escribir datos. La ejecución actual lee las fuentes completas; los watermarks del YAML todavía no se aplican.

In [ ]:
# Ajusta estos valores para cada ejecución.
LAKEHOUSE_ROOT = "/lakehouse/default"
CONFIG_RELATIVE_ROOT = "Files/CanonicalModel/config"
ENTITY_NAME = None  # None procesa todas las entidades habilitadas.
EXECUTION_MODE = None  # None usa persistence.strategy del YAML; o FULL_LOAD, UPSERT, SCD2.
PRECHECK_ONLY = True  # Cambia a False para permitir escrituras.

In [ ]:
from pathlib import Path

from fabric_canonical_model.exceptions import ConfigurationError, EntityResolutionError
from fabric_canonical_model.lib_audit import ExecutionAudit
from fabric_canonical_model.lib_config import FrameworkConfig
from fabric_canonical_model.lib_engine import CanonicalModelEngine
from fabric_canonical_model.lib_execution import EntityExecutionManager
from fabric_canonical_model.lib_mapping import AttributeMapper
from fabric_canonical_model.lib_persistence import CanonicalPersistence
from fabric_canonical_model.lib_registry import CanonicalRegistry
from fabric_canonical_model.lib_resolution import EntityResolver
from fabric_canonical_model.lib_source import SourceReader
from fabric_canonical_model.lib_transformations import TransformationEngine
from fabric_canonical_model.lib_utils import get_spark, read_yaml_file
from fabric_canonical_model.lib_validation import DataValidator

spark = get_spark()
lakehouse_root = Path(LAKEHOUSE_ROOT)
FRAMEWORK_FILE = lakehouse_root / CONFIG_RELATIVE_ROOT / "framework.yml"

if EXECUTION_MODE not in {None, "FULL_LOAD", "UPSERT", "SCD2"}:
    raise ValueError("EXECUTION_MODE debe ser None, FULL_LOAD, UPSERT o SCD2.")
if not isinstance(PRECHECK_ONLY, bool):
    raise ValueError("PRECHECK_ONLY debe ser booleano.")
if ENTITY_NAME is not None and (not isinstance(ENTITY_NAME, str) or not ENTITY_NAME.strip()):
    raise ValueError("ENTITY_NAME debe ser None o un nombre no vacío.")

## 1. Cargar configuración y registry

Usa la estructura de configuración dentro del Lakehouse adjunto. Para otro proyecto, despliega sus YAML en la misma estructura y cambia `CONFIG_RELATIVE_ROOT` solo si la carpeta es distinta.

In [ ]:
framework = FrameworkConfig.from_yaml(str(FRAMEWORK_FILE))
registry_file = lakehouse_root / framework.get_path("registry")
registry = CanonicalRegistry(str(registry_file))

enabled_entities = registry.get_enabled_entities()
if ENTITY_NAME is not None:
    enabled_entities = [item for item in enabled_entities if item["name"] == ENTITY_NAME]
    if not enabled_entities:
        raise ValueError(f"La entidad '{ENTITY_NAME}' no está habilitada en registry.yml")

if not enabled_entities:
    raise ValueError("El registry no contiene entidades habilitadas.")

class SelectedRegistry:
    def __init__(self, entities):
        self.entities = entities

    def get_enabled_entities(self):
        return self.entities

selected_registry = SelectedRegistry(enabled_entities)

enabled_entities

## 2. PRECHECK de configuraciones y fuentes

El precheck de este ejemplo comprueba la estructura que consumen los módulos actuales, referencias de fuentes, atributos requeridos y columnas físicas, incluidas todas las columnas de fallback. Todas las entidades seleccionadas superan esta fase antes de crear tablas de auditoría o destino. No reemplaza un validador JSON Schema completo del paquete.

In [ ]:
reader = SourceReader(spark)
KNOWN_OPERATIONS = TransformationEngine.SUPPORTED_OPERATIONS
TRANSFORMATION_FIELDS = {
    "TRIM": {"column"}, "UPPER": {"column"}, "LOWER": {"column"},
    "INITCAP": {"column"}, "CAST": {"column", "data_type"},
    "RENAME": {"source_column", "target_column"}, "DROP": {"columns"},
    "FILL_NULL": {"column", "value"}, "FILTER": {"condition"},
    "DERIVE_SQL": {"column", "expression"}
}

def precheck_entity(registry_entry):
    entity_name = registry_entry["name"]
    config_path = registry.get_entity_config_path(entity_name)
    config = read_yaml_file(config_path)
    if config.get("name") != entity_name:
        raise ConfigurationError(
            f"{config_path}: name debe coincidir con '{entity_name}'."
        )

    sources = config.get("sources")
    keys = config.get("business_keys")
    attributes = config.get("attributes")
    persistence = config.get("persistence")
    if not isinstance(sources, list) or not sources:
        raise ConfigurationError(f"{config_path}: sources debe ser una lista no vacía.")
    if not isinstance(keys, list) or not keys or any(not isinstance(key, str) or not key for key in keys):
        raise ConfigurationError(f"{config_path}: business_keys debe contener claves válidas.")
    if len(keys) != len(set(keys)):
        raise ConfigurationError(f"{config_path}: business_keys contiene duplicados.")
    if not isinstance(attributes, list) or not attributes:
        raise ConfigurationError(f"{config_path}: attributes debe ser una lista no vacía.")
    if not isinstance(persistence, dict) or persistence.get("strategy") not in {"FULL_LOAD", "UPSERT", "SCD2"}:
        raise ConfigurationError(f"{config_path}: persistence.strategy no es válido.")
    if not isinstance(persistence.get("table_name"), str) or not persistence["table_name"].strip():
        raise ConfigurationError(f"{config_path}: falta persistence.table_name.")

    source_names = [source.get("name") for source in sources if isinstance(source, dict)]
    if len(source_names) != len(sources) or any(not isinstance(name, str) or not name for name in source_names):
        raise ConfigurationError(f"{config_path}: cada fuente debe tener un name válido.")
    if len(source_names) != len(set(source_names)):
        raise ConfigurationError(f"{config_path}: hay nombres de fuente duplicados.")
    source_name_set = set(source_names)
    source_deduplication = {
        source["name"]: source["deduplication"]
        for source in sources
        if "deduplication" in source
    }
    try:
        source_deduplication = EntityResolver.validate_source_deduplication_config(
            source_names,
            source_deduplication
        )
    except EntityResolutionError as exc:
        raise ConfigurationError(
            f"{config_path}: configuración deduplication inválida: {exc}"
        ) from exc

    source_mapping = config.get("source_mapping", {})
    if not isinstance(source_mapping, dict) or not set(source_mapping).issubset(source_name_set):
        raise ConfigurationError(f"{config_path}: source_mapping referencia fuentes no declaradas.")
    for source_name, key_mapping in source_mapping.items():
        if not isinstance(key_mapping, dict) or not set(key_mapping).issubset(set(keys)):
            raise ConfigurationError(f"{config_path}: source_mapping.{source_name} debe mapear claves declaradas.")

    attribute_names = set()
    for index, attribute in enumerate(attributes):
        path = f"{config_path}: attributes[{index}]"
        if not isinstance(attribute, dict):
            raise ConfigurationError(f"{path} debe ser un objeto.")
        name = attribute.get("name")
        mappings = attribute.get("mapping")
        if not isinstance(name, str) or not name or name in attribute_names or name in keys:
            raise ConfigurationError(f"{path}.name es inválido, está duplicado o coincide con una clave.")
        if not isinstance(mappings, dict) or not mappings or not set(mappings).issubset(source_name_set):
            raise ConfigurationError(f"{path}.mapping debe referenciar fuentes declaradas.")
        if attribute.get("strategy") not in EntityResolver.SUPPORTED_STRATEGIES:
            raise ConfigurationError(f"{path}.strategy no está soportada.")
        priority = attribute.get("priority", [])
        if attribute["strategy"] == "SOURCE_PRIORITY" and not priority:
            raise ConfigurationError(f"{path}.priority es obligatorio para SOURCE_PRIORITY.")
        if not isinstance(priority, list) or not set(priority).issubset(source_name_set):
            raise ConfigurationError(f"{path}.priority contiene fuentes desconocidas.")
        attribute_names.add(name)

    transformations = config.get("transformations", [])
    if not isinstance(transformations, list):
        raise ConfigurationError(f"{config_path}: transformations debe ser una lista.")
    for index, transformation in enumerate(transformations):
        operation = transformation.get("operation") if isinstance(transformation, dict) else None
        if operation not in KNOWN_OPERATIONS:
            raise ConfigurationError(f"{config_path}: transformations[{index}].operation no está soportada.")
        missing = TRANSFORMATION_FIELDS[operation] - set(transformation)
        if missing:
            raise ConfigurationError(f"{config_path}: transformations[{index}] requiere {sorted(missing)}.")

    validations = config.get("validations", {})
    if not isinstance(validations, dict):
        raise ConfigurationError(f"{config_path}: validations debe ser un objeto.")
    required_columns = validations.get("required_columns", [])
    available_columns = set(keys) | attribute_names
    if not isinstance(required_columns, list) or not set(required_columns).issubset(available_columns):
        raise ConfigurationError(f"{config_path}: required_columns referencia atributos no declarados.")
    for rule in validations.get("custom_rules", []):
        if not isinstance(rule, dict) or not isinstance(rule.get("name"), str) or not isinstance(rule.get("condition"), str):
            raise ConfigurationError(f"{config_path}: cada custom_rule requiere name y condition.")

    physical_schemas = {}
    for source in sources:
        for field in ("schema", "table"):
            if not isinstance(source.get(field), str) or not source[field].strip():
                raise ConfigurationError(f"{config_path}: sources.{field} es obligatorio.")
        source_name = source["name"]
        physical_schemas[source_name] = reader.get_table_schema(source)
        required_physical = {
            source_mapping.get(source_name, {}).get(key, key) for key in keys
        }
        for attribute in attributes:
            physical_column = attribute["mapping"].get(source_name)
            if isinstance(physical_column, str):
                physical_columns = [physical_column]
            elif isinstance(physical_column, list) and physical_column and all(
                isinstance(column, str) and column.strip()
                for column in physical_column
            ):
                physical_columns = physical_column
            elif physical_column is None:
                physical_columns = []
            else:
                raise ConfigurationError(
                    f"{config_path}: attributes.{attribute.get('name')}.mapping.{source_name} "
                    "debe ser un nombre de columna o una lista ordenada de nombres."
                )
            if len(physical_columns) != len(set(physical_columns)):
                raise ConfigurationError(
                    f"{config_path}: attributes.{attribute.get('name')}.mapping.{source_name} "
                    "contiene columnas repetidas."
                )
            required_physical.update(physical_columns)
        watermark = source.get("watermark")
        if watermark:
            if not isinstance(watermark, dict) or not isinstance(watermark.get("column"), str):
                raise ConfigurationError(f"{config_path}: sources.{source_name}.watermark.column es obligatorio.")
            required_physical.add(watermark["column"])
        missing_columns = required_physical - set(physical_schemas[source_name])
        if missing_columns:
            raise ConfigurationError(
                f"{config_path}: la fuente '{source_name}' no tiene columnas: {sorted(missing_columns)}"
            )
        for key in keys:
            physical_key = source_mapping.get(source_name, {}).get(key, key)
            if physical_key != key and key in physical_schemas[source_name]:
                raise ConfigurationError(
                    f"{config_path}: '{source_name}' contiene simultáneamente '{key}' y '{physical_key}'."
                )

    return config, source_names

prechecked = {}
for entry in enabled_entities:
    prechecked[entry["name"]] = precheck_entity(entry)

print(f"PRECHECK exitoso: {len(prechecked)} entidad(es), fuentes y columnas verificadas.")

## 3. Preparar la función de procesamiento

Este adaptador conecta la configuración YAML con los componentes disponibles. Antes de resolver atributos, aplica las reglas de deduplicación por fuente. En esta versión, las fuentes se leen completas; no avanzar watermarks evita confirmar incrementos que aún no se procesan.

In [ ]:
def make_process_function(entity_config, execution_mode):
    def process_entity(execution_id):
        source_frames = reader.read_sources(entity_config["sources"])
        source_mapping = entity_config.get("source_mapping", {})
        source_deduplication = {
            source["name"]: source["deduplication"]
            for source in entity_config["sources"]
            if "deduplication" in source
        }
        normalized_frames = {}
        for source_name, dataframe in source_frames.items():
            for business_key, source_column in source_mapping.get(source_name, {}).items():
                if source_column != business_key:
                    dataframe = dataframe.withColumnRenamed(source_column, business_key)
            normalized_frames[source_name] = dataframe

        mapped_frames = AttributeMapper().map_sources(
            dataframes=normalized_frames,
            attributes_config=entity_config["attributes"],
            include_unmapped=True
        )
        resolver = EntityResolver()
        mapped_frames = resolver.deduplicate_sources(
            dataframes=mapped_frames,
            business_keys=entity_config["business_keys"],
            source_deduplication=source_deduplication
        )
        canonical_df = resolver.resolve_entity(
            dataframes=mapped_frames,
            business_keys=entity_config["business_keys"],
            attributes_config=entity_config["attributes"]
        )
        canonical_df = TransformationEngine().apply(
            canonical_df,
            entity_config.get("transformations", [])
        )

        validator = DataValidator()
        validation_config = entity_config.get("validations", {})
        quality = validator.validate_dataframe(
            dataframe=canonical_df,
            business_keys=entity_config["business_keys"],
            required_columns=validation_config.get("required_columns", []),
            check_duplicates=validation_config.get("check_duplicates", True)
        )
        rule_results = validator.validate_custom_rules(
            canonical_df,
            validation_config.get("custom_rules", [])
        )
        if not quality["passed"] or any(not result["passed"] for result in rule_results):
            raise ValueError(
                f"La entidad '{entity_config['name']}' no superó las validaciones: "
                f"quality={quality}, rules={rule_results}"
            )

        records_read = sum(dataframe.count() for dataframe in source_frames.values())
        persistence_config = entity_config["persistence"]
        persistence = CanonicalPersistence(
            spark=spark,
            target_schema=framework.get_schema("canonical")
        )
        persist = getattr(persistence, execution_mode.lower())
        persist(
            dataframe=canonical_df,
            table_name=persistence_config["table_name"],
            business_keys=entity_config["business_keys"]
        )
        return {
            "metrics": {"records_read": records_read},
            "candidate_watermarks": {}
        }

    return process_entity

entity_runtime = {}
for entity_name, (entity_config, source_names) in prechecked.items():
    selected_mode = EXECUTION_MODE or entity_config["persistence"]["strategy"]
    entity_runtime[entity_name] = {
        "execution_mode": selected_mode,
        "source_names": source_names,
        "process_function": make_process_function(entity_config, selected_mode),
        "process_name": "nb_create_canonical_model"
    }

print("Callbacks preparados:", list(entity_runtime))

## 4. Ejecutar bajo demanda

La escritura solo ocurre cuando `PRECHECK_ONLY = False`. Revise los schemas y nombres de tabla del YAML antes de habilitarla.

In [ ]:
if PRECHECK_ONLY:
    print("PRECHECK exitoso. No se realizaron escrituras. Cambia PRECHECK_ONLY = False para ejecutar.")
else:
    audit = ExecutionAudit(
        spark=spark,
        schema_name=framework.get_schema("metadata")
    )
    execution_manager = EntityExecutionManager(
        audit_manager=audit,
        watermark_manager=None
    )
    engine = CanonicalModelEngine(
        registry=selected_registry,
        execution_manager=execution_manager
    )
    result = engine.execute_enabled_entities(
        entity_runtime=entity_runtime,
        stop_on_error=True
    )
    display(result)
    if result["failed_entities"]:
        raise RuntimeError(f"Falló la ejecución: {result}")

### Límites de esta versión

- Aunque los YAML declaren `watermark`, este notebook todavía lee todas las filas y no confirma watermarks.
- La validación estructural incluida aquí es un precheck de ejemplo, no sustituye el validador central obligatorio del framework.
- El precheck físico consulta schemas de tablas Curated y se ejecuta para todas las entidades seleccionadas antes de iniciar auditoría o persistencia.
- Las escrituras Delta solo ocurren si `PRECHECK_ONLY` se cambia explícitamente a `False`.